In [ ]:
import wandb
import pandas as pd
import numpy as np
import json
import re
import os
from pathlib import Path

# ============================================================
# Toggle: "1B" or "7B"
# ============================================================
MODEL_SIZE = "7B"

# ============================================================
# Per-size configuration
# ============================================================
HOME = os.environ['HOME']
NOTEBOOKS_DIR = (Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks')
SAVES_DIR = Path(HOME) / 'OLMoBenchOutputs' / 'saves'

CONFIG = {
    "1B": {
        "cache_path": NOTEBOOKS_DIR / ".wandb_cache.json",
        "output_dir": SAVES_DIR / "Train_95frozen_FullSubset",
        "wandb_prefix": "OLMo_Mask_Unlearn",
        "model_label": "Masked",
    },
    "7B": {
        "cache_path": NOTEBOOKS_DIR / ".wandb_cache_7B.json",
        "output_dir": SAVES_DIR / "7B_Train_95frozen_FullSubset",
        "wandb_prefix": "OLMo7_Mask_Unlearn",
        "model_label": "Masked",
    },
}

cfg = CONFIG[MODEL_SIZE]
CACHE_PATH = cfg["cache_path"]
PRECISION_OUTPUT_DIR = cfg["output_dir"]
WANDB_PREFIX = cfg["wandb_prefix"]
MODEL = cfg["model_label"]

PROJECT = "siva-reddy-mila-org/unlearning-bench"
FIELDS = ["Email_Address", "Phone_Number", "Birth_City", "Drivers_License"]
METHODS = ["SimNPO", "MemFlex", "AlphaEdit", "OracleGrad"]

PANORAMA_KEYS = [
    "panorama/exact_memorization", "panorama/exact_memorization_paraphrased",
    "panorama/extraction_strength", "panorama/extraction_strength_paraphrased",
    "panorama/forget_Q_A_Prob", "panorama/forget_Q_A_Prob_paraphrased",
    "panorama/forget_Q_A_ROUGE", "panorama/forget_Q_A_ROUGE_paraphrased",
    "panorama/retain_exact_memorization", "panorama/retain_exact_memorization_paraphrased",
    "panorama/retain_extraction_strength", "panorama/retain_extraction_strength_paraphrased",
    "panorama/retain_Q_A_Prob", "panorama/retain_Q_A_Prob_paraphrased",
    "panorama/retain_Q_A_ROUGE", "panorama/retain_Q_A_ROUGE_paraphrased",
]
UTILITY_KEYS = [
    "utility_metrics/arc_challenge/acc,none",
    "utility_metrics/arc_easy/acc,none",
    "utility_metrics/hellaswag/acc,none",
    "utility_metrics/mmlu/acc,none",
]
ALL_KEYS = PANORAMA_KEYS + UTILITY_KEYS

USE_CACHE = True  # Set to False to force re-fetch from wandb

if USE_CACHE and CACHE_PATH.exists():
    df = pd.read_json(CACHE_PATH)
    print(f"Loaded {len(df)} results from cache ({CACHE_PATH})")
else:
    api = wandb.Api()
    all_runs = api.runs(PROJECT, per_page=200)

    # Deduplicate: prefer the most recent finished run
    run_index = {}
    for r in all_runs:
        if r.state == "finished":
            if r.name not in run_index or r.created_at > run_index[r.name].created_at:
                run_index[r.name] = r

    # Verify expected runs exist
    expected_names = [f"{WANDB_PREFIX}_{method}_{field}" for field in FIELDS for method in METHODS]
    found = [n for n in expected_names if n in run_index]
    missing = [n for n in expected_names if n not in run_index]
    print(f"Found {len(found)}/{len(expected_names)} expected runs.")
    if missing:
        print(f"Missing runs (will show as ---): {missing}")

    def get_run_data(run, step=None):
        if step is None:
            return {k: run.summary.get(k) for k in ALL_KEYS}
        else:
            for row in run.scan_history():
                if row.get("global_step") == step:
                    return {k: row.get(k) for k in ALL_KEYS}
        return None

    rows = []
    for field in FIELDS:
        # Baseline from SimNPO step 0
        run_name = f"{WANDB_PREFIX}_SimNPO_{field}"
        if run_name in run_index:
            baseline = get_run_data(run_index[run_name], step=0)
            if baseline:
                rows.append({"Model": MODEL, "Method": "Baseline", "Field": field, **baseline})
        for method in METHODS:
            run_name = f"{WANDB_PREFIX}_{method}_{field}"
            if run_name in run_index:
                data = get_run_data(run_index[run_name])
                if data:
                    rows.append({"Model": MODEL, "Method": method, "Field": field, **data})
            else:
                print(f"WARNING: Run not found: {run_name}")

    df = pd.DataFrame(rows)
    df.columns = [c.replace("panorama/", "").replace("utility_metrics/", "") for c in df.columns]

    # Save cache
    df.to_json(CACHE_PATH)
    print(f"Fetched {len(df)} results from wandb and cached to {CACHE_PATH}")

# --- Load precision metrics from cached files ---
PRECISION_CACHE_DIR = PRECISION_OUTPUT_DIR / "cached_notebook_files" / "precision_metrics"
PRECISION_MASK_TYPES = ["unified", "forget"]
PRECISION_ALL_METRICS = [
    "raw", "qtile", "compnorm", "contrast", "contrastnorm", "contrastln",
    "signrev", "layernorm", "reversal", "dirreversal", "eratio", "crossfield", "composite",
]
UNMASK_METRICS = {"compnorm", "contrast", "contrastnorm", "contrastln", "eratio"}
UNMASK_EXCLUDED_METHODS = {"OracleGrad"}
FORGET_ONLY_METHODS = {"OracleGrad"}

# precision_auc[mask_type][field][method] = best_auc (float)
precision_auc = {}
for mask_type in PRECISION_MASK_TYPES:
    precision_auc[mask_type] = {}
    for field in FIELDS:
        precision_auc[mask_type][field] = {}
        for method in METHODS:
            if mask_type == "unified" and method in FORGET_ONLY_METHODS:
                continue
            metrics_path = PRECISION_CACHE_DIR / mask_type / field / method / "metrics.json"
            if not metrics_path.exists():
                continue
            with open(metrics_path) as f:
                metrics = json.load(f)
            best_auc = -1
            for m in PRECISION_ALL_METRICS:
                if m in UNMASK_METRICS and method in UNMASK_EXCLUDED_METHODS:
                    continue
                v = metrics.get(f"auc_{m}", float("nan"))
                if not np.isnan(v) and v > best_auc:
                    best_auc = v
            if best_auc > 0:
                precision_auc[mask_type][field][method] = best_auc

print(f"Model size: {MODEL_SIZE}")
print(f"Precision AUC loaded: {sum(len(v) for mt in precision_auc.values() for v in mt.values())} entries")
df.head()

In [ ]:
# Metric groups with multi-level column headers: (group, subcolumn) -> df column
FORGET_METRICS = {
    "EM": "exact_memorization",
    "ES": "extraction_strength",
    "EM Paraph.": "exact_memorization_paraphrased",
    "ES Paraph.": "extraction_strength_paraphrased",
    "Prob": "forget_Q_A_Prob",
    "Prob Paraph.": "forget_Q_A_Prob_paraphrased",
}

RETAIN_METRICS = {
    "EM": "retain_exact_memorization",
    "ES": "retain_extraction_strength",
    "EM Paraph.": "retain_exact_memorization_paraphrased",
    "ES Paraph.": "retain_extraction_strength_paraphrased",
    "Prob": "retain_Q_A_Prob",
    "Prob Paraph.": "retain_Q_A_Prob_paraphrased",
}

UTILITY_METRICS = {
    "ARC-C": "arc_challenge/acc,none",
    "ARC-E": "arc_easy/acc,none",
    "HSwag": "hellaswag/acc,none",
    "MMLU": "mmlu/acc,none",
}

# Precision metrics are special: sourced from precision_auc dict, not from df columns
PRECISION_METRICS = {
    "AUC (F|R)": "precision_unified",   # best AUC with unified (forget|retain) GT mask
    "AUC (F)": "precision_forget",      # best AUC with forget-only GT mask
}

ALL_METRIC_GROUPS = [
    ("Forget", FORGET_METRICS),
    ("Retain", RETAIN_METRICS),
    ("Utility", UTILITY_METRICS),
    ("Precision", PRECISION_METRICS),
]

In [ ]:
def get_precision_value(field, method, df_col):
    """Look up precision AUC from the precision_auc dict."""
    if df_col == "precision_unified":
        return precision_auc.get("unified", {}).get(field, {}).get(method)
    elif df_col == "precision_forget":
        return precision_auc.get("forget", {}).get(field, {}).get(method)
    return None


def make_latex_table(df_field, field_name, metric_groups=None):
    """Generate a LaTeX table: metrics on rows, methods on columns.
    Bold best method per metric (excluding Original Model).
    Forget: lower is better. Retain/Utility/Precision: higher is better."""
    if metric_groups is None:
        metric_groups = ALL_METRIC_GROUPS
    model = "Masked"
    methods = ["Baseline", "AlphaEdit", "MemFlex", "OracleGrad", "SimNPO"]
    method_labels = {"Baseline": "Original Model", "AlphaEdit": "AlphaEdit",
                     "MemFlex": "MemFlex", "OracleGrad": "OracleGrad", "SimNPO": "SimNPO"}
    
    field = field_name  # used for precision lookup
    
    row_tuples = []
    row_keys = []
    row_groups = []
    for group_name, metrics in metric_groups:
        for sub_name, df_col in metrics.items():
            row_tuples.append((group_name, sub_name))
            row_keys.append(df_col)
            row_groups.append(group_name)
    
    # Collect raw numeric data
    raw_data = []
    for df_col in row_keys:
        row_vals = {}
        is_precision = df_col.startswith("precision_")
        for method in methods:
            if is_precision:
                # Precision has no Baseline
                if method == "Baseline":
                    row_vals[method] = None
                else:
                    row_vals[method] = get_precision_value(field, method, df_col)
            else:
                mask = (df_field["Model"] == model) & (df_field["Method"] == method)
                matched = df_field.loc[mask]
                if len(matched) > 0:
                    v = matched.iloc[0][df_col]
                    row_vals[method] = v if pd.notna(v) else None
                else:
                    row_vals[method] = None
        raw_data.append(row_vals)
    
    # Format with bolding
    data = []
    for row_idx, (df_col, group) in enumerate(zip(row_keys, row_groups)):
        row_vals_raw = raw_data[row_idx]
        lower_is_better = (group == "Forget")
        is_precision = df_col.startswith("precision_")
        
        candidate_vals = []
        for method in methods:
            if method == "Baseline":
                continue
            v = row_vals_raw.get(method)
            if v is not None:
                candidate_vals.append(v)
        best_val = (min(candidate_vals) if lower_is_better else max(candidate_vals)) if candidate_vals else None
        
        row_formatted = []
        for method in methods:
            v = row_vals_raw.get(method)
            if v is None:
                row_formatted.append("---")
            else:
                if is_precision:
                    formatted = f"{v:.3f}"
                else:
                    formatted = f"{v*100:.1f}"
                if method != "Baseline" and best_val is not None and abs(v - best_val) < 1e-9:
                    formatted = f"\\textbf{{{formatted}}}"
                row_formatted.append(formatted)
        data.append(row_formatted)
    
    col_index = [method_labels[m] for m in methods]
    row_index = pd.MultiIndex.from_tuples(row_tuples, names=["", ""])
    display_df = pd.DataFrame(data, index=row_index, columns=col_index)
    
    field_label = field_name.replace("_", " ")
    
    col_fmt = "ll" + "r" * len(methods)
    latex = display_df.to_latex(
        caption=f"Unlearning Results --- {field_label}",
        label=f"tab:{field_name.lower()}",
        column_format=col_fmt,
        multirow=True,
        multicolumn=True,
        multicolumn_format="c",
        escape=False,
    )
    
    # Remove empty header row
    lines = latex.split("\n")
    cleaned = []
    for line in lines:
        stripped = line.strip()
        if re.match(r'^(&\s*)+\\\\$', stripped):
            continue
        cleaned.append(line)
    latex = "\n".join(cleaned)
    
    # Replace \cline with \cmidrule
    latex = re.sub(r'\\cline\{(\d+-\d+)\}', r'\\cmidrule{\1}', latex)
    
    # Wrap tabular in adjustbox for centering
    latex = latex.replace(
        "\\begin{tabular}",
        "\\begin{adjustbox}{center}\n\\begin{tabular}"
    )
    latex = latex.replace(
        "\\end{tabular}",
        "\\end{tabular}\n\\end{adjustbox}"
    )
    
    # Add centering and small font
    latex = latex.replace("\\begin{table}", "\\begin{table}[htbp]\n\\centering\\small", 1)
    
    return latex, display_df

In [ ]:
# Generate tables for each field
all_latex = []
for field in FIELDS:
    df_field = df[df["Field"] == field].copy()
    # Sort: Baseline first, then methods alphabetically, within each model
    method_order = {"Baseline": 0, "AlphaEdit": 1, "MemFlex": 2, "OracleGrad": 3, "SimNPO": 4}
    df_field["_sort"] = df_field["Method"].map(method_order)
    df_field = df_field.sort_values(["Model", "_sort"]).drop(columns=["_sort"])
    
    latex, display_df = make_latex_table(df_field, field)
    all_latex.append(latex)
    
    field_label = field.replace("_", " ")
    print(f"\n{'='*80}")
    print(f"  {field_label}")
    print(f"{'='*80}")
    display(display_df)
    print(latex)

In [ ]:
# Save all per-field tables to a single .tex file
output_path = NOTEBOOKS_DIR / f"results_tables_{MODEL_SIZE}.tex"
with open(output_path, "w") as f:
    for latex in all_latex:
        f.write(latex)
        f.write("\n\n")
print(f"Saved {len(all_latex)} per-field tables to {output_path}")

In [ ]:
# Master table: metrics as rows (grouped by category), fields as columns with methods as sub-columns

FIELD_ABBREV = {
    "Email_Address": "Email",
    "Phone_Number": "Phone",
    "Birth_City": "Birth City",
    "Drivers_License": "Driver's Lic.",
}
METHOD_ABBREV = {"AlphaEdit": "AE", "MemFlex": "MF", "OracleGrad": "OG", "SimNPO": "SN"}

# Full field names (used for the 1B master table header)
FIELD_HEADER_FULL = {
    "Email_Address": "Email Address",
    "Phone_Number": "Phone Number",
    "Birth_City": "Birth City",
    "Drivers_License": "Driver's License",
}

# Per-model-size LaTeX presentation settings (to match the paper exactly)
MASTER_TABLE_STYLE = {
    "1B": {"olmo_macro": "\\olmosmall", "minipage_opt": "",    "field_header": FIELD_HEADER_FULL, "label": "tab:master_all_fields-1B"},
    "7B": {"olmo_macro": "\\olmobig",   "minipage_opt": "[t]", "field_header": FIELD_ABBREV,       "label": "tab:master_all_fields-7B"},
}

def make_master_table(df):
    """Master table: rows = (Category, Metric), columns = (Field, Method)."""
    model = MODEL
    all_methods = ["Baseline", "AlphaEdit", "MemFlex", "OracleGrad", "SimNPO"]
    display_methods = ["AlphaEdit", "MemFlex", "OracleGrad", "SimNPO"]

    # Build row specs
    row_specs = []
    for group_name, metrics in ALL_METRIC_GROUPS:
        for sub_name, df_col in metrics.items():
            row_specs.append((group_name, sub_name, df_col))

    # Collect raw values (including Baseline for delta computation)
    raw = {}
    for method in all_methods:
        for field in FIELDS:
            df_field = df[(df["Field"] == field) & (df["Model"] == model) & (df["Method"] == method)]
            for group, sub, df_col in row_specs:
                is_precision = df_col.startswith("precision_")
                if is_precision:
                    v = None if method == "Baseline" else get_precision_value(field, method, df_col)
                else:
                    if len(df_field) > 0:
                        v = df_field.iloc[0][df_col]
                        v = v if pd.notna(v) else None
                    else:
                        v = None
                raw[(method, group, sub, field)] = v

    def get_display_value(method, group, sub, df_col, field):
        v = raw[(method, group, sub, field)]
        if v is None:
            return None
        if group == "Utility":
            baseline_v = raw.get(("Baseline", group, sub, field))
            if baseline_v is not None:
                return v - baseline_v
            return None
        return v

    # Best per (metric, field) among display methods
    best = {}
    for group, sub, df_col in row_specs:
        lower_is_better = (group == "Forget")
        for field in FIELDS:
            cands = [get_display_value(m, group, sub, df_col, field)
                     for m in display_methods
                     if get_display_value(m, group, sub, df_col, field) is not None]
            if cands:
                best[(group, sub, field)] = min(cands) if lower_is_better else max(cands)

    def fmt(method, group, sub, df_col, field):
        v = get_display_value(method, group, sub, df_col, field)
        if v is None:
            return "\\multicolumn{1}{c}{---}"
        is_precision = df_col.startswith("precision_")
        is_delta = (group == "Utility")
        if is_precision:
            s = f"{v:.3f}"
        elif is_delta:
            val = v * 100
            sign = "+" if val >= 0 else ""
            s = f"{sign}{val:.1f}"
        else:
            s = f"{v*100:.1f}"
        bv = best.get((group, sub, field))
        if bv is not None and abs(v - bv) < 1e-9:
            s = f"\\textbf{{{s}}}"
        return s

    # --- Build LaTeX manually ---
    n_methods = len(display_methods)
    n_fields = len(FIELDS)
    # ll | mmmm | mmmm | mmmm | mmmm  (one group of methods per field)
    col_fmt = "ll" + ("|" + "r" * n_methods) * n_fields

    style = MASTER_TABLE_STYLE[MODEL_SIZE]
    field_header = style["field_header"]

    lines = []
    lines.append(f"\\begin{{minipage}}{style['minipage_opt']}{{\\textwidth}}")
    lines.append("\\centering\\footnotesize")
    lines.append(
        f"\\captionof{{table}}{{Unlearning Results - {style['olmo_macro']} - "
        "Cumulative results for all unlearning methods: "
        "\\textbf{AlphaEdit} (AE), \\textbf{MemFlex} (MF), "
        "\\textbf{OracleGrad} (OG), \\textbf{SimNPO} (SN).}"
    )
    lines.append(f"\\label{{{style['label']}}}")
    lines.append("\\begin{adjustbox}{max width=\\textwidth}")
    lines.append("\\setlength{\\tabcolsep}{3pt}")
    lines.append(f"\\begin{{tabular}}{{{col_fmt}}}")
    lines.append("\\toprule")

    # Header row 1: field names spanning methods each
    hdr1_parts = [" & "]
    for field in FIELDS:
        hdr1_parts.append(f"\\multicolumn{{{n_methods}}}{{c}}{{{field_header[field]}}}")
    lines.append(" & ".join(hdr1_parts) + " \\\\")

    # Header row 2: method abbreviations repeated per field
    hdr2_parts = [" & "]
    for field in FIELDS:
        for method in display_methods:
            hdr2_parts.append(METHOD_ABBREV[method])
    lines.append(" & ".join(hdr2_parts) + " \\\\")
    lines.append("\\midrule")

    # Data rows
    prev_group = None
    group_counts = {}
    for group, sub, df_col in row_specs:
        group_counts[group] = group_counts.get(group, 0) + 1

    group_emitted = {}
    for group, sub, df_col in row_specs:
        if prev_group is not None and group != prev_group:
            lines.append("\\midrule")

        group_display = group
        if group == "Utility":
            group_display = f"{group} ($\\Delta$)"

        if group not in group_emitted:
            row_label = f"\\multirow{{{group_counts[group]}}}{{*}}{{{group_display}}} & {sub}"
            group_emitted[group] = True
        else:
            row_label = f" & {sub}"

        cells = [row_label]
        for field in FIELDS:
            for method in display_methods:
                cells.append(fmt(method, group, sub, df_col, field))
        lines.append(" & ".join(cells) + " \\\\")
        prev_group = group

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{adjustbox}")
    lines.append("\\end{minipage}")

    latex = "\n".join(lines)

    # Display DataFrame
    col_tuples = []
    for field in FIELDS:
        for method in display_methods:
            col_tuples.append((FIELD_ABBREV[field], METHOD_ABBREV[method]))
    col_index = pd.MultiIndex.from_tuples(col_tuples, names=["Field", "Method"])
    row_tuples = [(g, s) for g, s, _ in row_specs]
    row_index = pd.MultiIndex.from_tuples(row_tuples, names=["", ""])

    display_data = []
    for group, sub, df_col in row_specs:
        row = []
        for field in FIELDS:
            for method in display_methods:
                row.append(fmt(method, group, sub, df_col, field))
        display_data.append(row)
    display_df = pd.DataFrame(display_data, index=row_index, columns=col_index)

    return latex, display_df

master_latex, master_df = make_master_table(df)

print("=" * 80)
print(f"  MASTER TABLE ({MODEL_SIZE})")
print("=" * 80)
display(master_df)
print(master_latex)

# Save master table
master_path = NOTEBOOKS_DIR / f"master_results_table_{MODEL_SIZE}.tex"
with open(master_path, "w") as f:
    f.write(master_latex)
print(f"\nSaved master table to {master_path}")

In [ ]:
# ============================================================
# Combined master table -> single file notebooks/mastertable.tex
# Emits both model sizes (1B then 7B), each as a \minipage, matching
# the per-size make_master_table() output. Independent of the MODEL_SIZE
# toggle above: it loads each size's cache + precision data directly.
# ============================================================
def _load_precision_auc(output_dir):
    cache_dir = output_dir / "cached_notebook_files" / "precision_metrics"
    mask_types = ["unified", "forget"]
    all_metrics = [
        "raw", "qtile", "compnorm", "contrast", "contrastnorm", "contrastln",
        "signrev", "layernorm", "reversal", "dirreversal", "eratio", "crossfield", "composite",
    ]
    unmask_metrics = {"compnorm", "contrast", "contrastnorm", "contrastln", "eratio"}
    unmask_excluded_methods = {"OracleGrad"}
    forget_only_methods = {"OracleGrad"}
    pa = {}
    for mask_type in mask_types:
        pa[mask_type] = {}
        for field in FIELDS:
            pa[mask_type][field] = {}
            for method in METHODS:
                if mask_type == "unified" and method in forget_only_methods:
                    continue
                metrics_path = cache_dir / mask_type / field / method / "metrics.json"
                if not metrics_path.exists():
                    continue
                with open(metrics_path) as f:
                    metrics = json.load(f)
                best_auc = -1
                for m in all_metrics:
                    if m in unmask_metrics and method in unmask_excluded_methods:
                        continue
                    v = metrics.get(f"auc_{m}", float("nan"))
                    if not np.isnan(v) and v > best_auc:
                        best_auc = v
                if best_auc > 0:
                    pa[mask_type][field][method] = best_auc
    return pa

# Preserve current globals (make_master_table / get_precision_value read these)
_orig_size, _orig_model, _orig_pa = MODEL_SIZE, MODEL, precision_auc

_combined = []
for _size in ["1B", "7B"]:
    _cfg = CONFIG[_size]
    _df = pd.read_json(_cfg["cache_path"])
    MODEL_SIZE = _size
    MODEL = _cfg["model_label"]
    precision_auc = _load_precision_auc(_cfg["output_dir"])
    _latex, _ = make_master_table(_df)
    _combined.append(_latex)
    print(f"  built {_size} master table ({len(_df)} rows from {_cfg['cache_path'].name})")

# Restore globals
MODEL_SIZE, MODEL, precision_auc = _orig_size, _orig_model, _orig_pa

combined_latex = "\n\n".join(_combined) + "\n"
combined_path = NOTEBOOKS_DIR / "mastertable.tex"
with open(combined_path, "w") as f:
    f.write(combined_latex)
print(f"\nWrote combined master table (1B + 7B) to {combined_path}")
